# Week 8: Multi-Agent Deal Hunting (Mock Pipeline)

This notebook showcases a **mock multi-agent deal discovery loop** with a live UI:
- Specialized agents (scanner, pricer/ensemble, planner, notifier)
- Streaming logs + periodic scans using Gradio + threads/queues
- Persistent “memory” of flagged opportunities (JSON on disk)
- Simple 3D vector-space visualization (random demo data)

> Note: This is a demo-only scaffold—swap the mock agents with real RSS scraping, RAG (Chroma), and production alerting as needed.

In [ ]:
!pip install -q gradio pydantic openai chromadb sentence-transformers scikit-learn feedparser beautifulsoup4 requests plotly

In [ ]:
import os
import logging
import queue
import threading
import time
import json
from typing import List, Optional
from datetime import datetime

import gradio as gr
import plotly.graph_objects as go
from pydantic import BaseModel

In [ ]:
class DealRecord(BaseModel):
    description: str
    price_usd: float
    link: str


class DealBatch(BaseModel):
    items: List[DealRecord]


class DealOpportunity(BaseModel):
    deal: DealRecord
    fair_value: float
    savings: float


class BaseAgent:
    label = "Base Agent"
    ansi_color = '\033[37m'

    def log(self, msg: str) -> None:
        logging.info(f"[{self.label}] {msg}")


class DemoScannerAgent(BaseAgent):
    label = "Scanner Agent"

    def scan(self, memory=None) -> DealBatch:
        self.log("Simulating RSS feed scan")
        time.sleep(1)

        fake_deals = [
            DealRecord(
                description=(
                    "Apple iPad Pro 11-inch 256GB WiFi (latest model) - Space Gray. "
                    "Features M2 chip, Liquid Retina display, 12MP camera, Face ID, and all-day battery life."
                ),
                price_usd=749.99,
                link="https://example.com/ipad",
            ),
            DealRecord(
                description=(
                    "Sony WH-1000XM5 Wireless Noise Cancelling Headphones - Industry-leading noise cancellation, "
                    "exceptional sound quality, 30-hour battery life, comfortable design."
                ),
                price_usd=329.99,
                link="https://example.com/sony-headphones",
            ),
        ]

        return DealBatch(items=fake_deals)


class DemoEnsembleAgent(BaseAgent):
    label = "Ensemble Agent"

    def estimate(self, description: str) -> float:
        self.log("Estimating price for product")
        time.sleep(0.5)

        if "iPad" in description:
            return 899.00
        if "Sony" in description:
            return 398.00
        return 150.00


class DemoNotifyAgent(BaseAgent):
    label = "Messaging Agent"

    def notify(self, opportunity: DealOpportunity) -> None:
        self.log(
            f"Alert sent: ${opportunity.savings:.2f} discount on {opportunity.deal.description[:50]}..."
        )


class DemoPlannerAgent(BaseAgent):
    label = "Planning Agent"
    MIN_SAVINGS = 50

    def __init__(self):
        self.scanner = DemoScannerAgent()
        self.pricer = DemoEnsembleAgent()
        self.notifier = DemoNotifyAgent()

    def plan_once(self, memory=None) -> Optional[DealOpportunity]:
        if memory is None:
            memory = []

        self.log("Starting planning cycle")
        batch = self.scanner.scan(memory)

        if not batch or not batch.items:
            return None

        opportunities: list[DealOpportunity] = []
        for deal in batch.items:
            fair_value = self.pricer.estimate(deal.description)
            savings = fair_value - deal.price_usd
            opportunities.append(DealOpportunity(deal=deal, fair_value=fair_value, savings=savings))

        opportunities.sort(key=lambda o: o.savings, reverse=True)
        best = opportunities[0]

        self.log(f"Best deal has discount: ${best.savings:.2f}")

        if best.savings > self.MIN_SAVINGS:
            self.notifier.notify(best)
            return best

        return None


class DemoDealFramework:
    STATE_PATH = "mock_memory.json"

    def __init__(self):
        self.memory = self._load_state()
        self.planner: Optional[DemoPlannerAgent] = None

    def ensure_ready(self) -> None:
        if self.planner is None:
            logging.info("Initializing Mock Agent Framework")
            self.planner = DemoPlannerAgent()
            logging.info("Mock Agent Framework ready")

    def _load_state(self) -> List[DealOpportunity]:
        if os.path.exists(self.STATE_PATH):
            try:
                with open(self.STATE_PATH, 'r') as f:
                    payload = json.load(f)
                return [DealOpportunity(**item) for item in payload]
            except Exception:
                return []
        return []

    def _save_state(self) -> None:
        payload = [opp.dict() for opp in self.memory]
        with open(self.STATE_PATH, 'w') as f:
            json.dump(payload, f, indent=2)

    def run_cycle(self) -> List[DealOpportunity]:
        self.ensure_ready()
        assert self.planner is not None

        result = self.planner.plan_once(memory=self.memory)
        if result:
            self.memory.append(result)
            self._save_state()

        return self.memory

    @classmethod
    def plot_payload(cls, max_points=100):
        import numpy as np

        n = min(100, max_points)
        points = np.random.randn(n, 3)
        labels = [f"Product {i}" for i in range(n)]
        palette = ['red', 'blue', 'green', 'orange'] * (n // 4 + 1)
        return labels[:n], points, palette[:n]

In [ ]:
ANSI_BG_BLACK = '\033[40m'
ANSI_RED = '\033[31m'
ANSI_GREEN = '\033[32m'
ANSI_YELLOW = '\033[33m'
ANSI_BLUE = '\033[34m'
ANSI_MAGENTA = '\033[35m'
ANSI_CYAN = '\033[36m'
ANSI_WHITE = '\033[37m'
ANSI_BG_BLUE = '\033[44m'
ANSI_RESET = '\033[0m'

ansi_to_hex = {
    ANSI_BG_BLACK + ANSI_RED: "#dd0000",
    ANSI_BG_BLACK + ANSI_GREEN: "#00dd00",
    ANSI_BG_BLACK + ANSI_YELLOW: "#dddd00",
    ANSI_BG_BLACK + ANSI_BLUE: "#0000ee",
    ANSI_BG_BLACK + ANSI_MAGENTA: "#aa00dd",
    ANSI_BG_BLACK + ANSI_CYAN: "#00dddd",
    ANSI_BG_BLACK + ANSI_WHITE: "#87CEEB",
    ANSI_BG_BLUE + ANSI_WHITE: "#ff7800",
}


def ansi_to_html(message: str) -> str:
    for key, value in ansi_to_hex.items():
        message = message.replace(key, f'<span style="color: {value}">')
    return message.replace(ANSI_RESET, '</span>')


class LogQueueHandler(logging.Handler):
    def __init__(self, q: queue.Queue):
        super().__init__()
        self.q = q

    def emit(self, record):
        self.q.put(self.format(record))


def init_logging(q: queue.Queue) -> None:
    handler = LogQueueHandler(q)
    handler.setFormatter(
        logging.Formatter("[%(asctime)s] %(message)s", datefmt="%Y-%m-%d %H:%M:%S")
    )
    root = logging.getLogger()
    root.addHandler(handler)
    root.setLevel(logging.INFO)


def logs_to_html(lines: list[str]) -> str:
    view = '<br>'.join(lines[-20:])
    return f"""
    <div style="height: 420px; overflow-y: auto; border: 1px solid #444; background-color: #1a1a1a; padding: 12px; font-family: monospace; font-size: 13px; color: #fff;">
        {view}
    </div>
    """


def build_plot() -> go.Figure:
    try:
        docs, points, colors = DemoDealFramework.plot_payload(max_points=100)

        fig = go.Figure(data=[go.Scatter3d(
            x=points[:, 0],
            y=points[:, 1],
            z=points[:, 2],
            mode='markers',
            marker=dict(size=3, color=colors, opacity=0.7),
        )])

        fig.update_layout(
            scene=dict(
                xaxis_title='X',
                yaxis_title='Y',
                zaxis_title='Z',
                aspectmode='manual',
                aspectratio=dict(x=2.2, y=2.2, z=1),
                camera=dict(eye=dict(x=1.6, y=1.6, z=0.8)),
                bgcolor='#1a1a1a',
            ),
            height=420,
            margin=dict(r=5, b=5, l=5, t=5),
            paper_bgcolor='#1a1a1a',
            font=dict(color='#ffffff'),
            title="Mock Vector Space (Random Data for Demo)",
        )

        return fig
    except Exception as e:
        fig = go.Figure()
        fig.update_layout(
            title=f"Error: {str(e)}",
            height=420,
            paper_bgcolor='#1a1a1a',
            font=dict(color='#ffffff'),
        )
        return fig

In [ ]:
class DealHunterUI:

    def __init__(self):
        self.framework: Optional[DemoDealFramework] = None

    def _framework(self) -> DemoDealFramework:
        if self.framework is None:
            self.framework = DemoDealFramework()
            self.framework.ensure_ready()
        return self.framework

    def _to_rows(self, opportunities: List[DealOpportunity]):
        if not opportunities:
            return []

        rows = []
        for opp in opportunities:
            if not isinstance(opp, DealOpportunity):
                continue
            rows.append([
                opp.deal.description,
                f"${opp.deal.price_usd:.2f}",
                f"${opp.fair_value:.2f}",
                f"${opp.savings:.2f}",
                opp.deal.link,
            ])
        return rows

    def scan(self):
        fw = self._framework()
        logging.info(
            f"Scan triggered at {datetime.now().strftime('%H:%M:%S')} - Current memory: {len(fw.memory)} deals"
        )
        updated = fw.run_cycle()
        logging.info(f"Scan complete - Total deals: {len(fw.memory)}")
        return self._to_rows(updated)

    def scan_stream(self, log_lines: list[str]):
        log_q: queue.Queue = queue.Queue()
        done_q: queue.Queue = queue.Queue()
        init_logging(log_q)

        def worker():
            done_q.put(self.scan())

        threading.Thread(target=worker).start()

        fw = self._framework()
        table = self._to_rows(fw.memory)

        while True:
            try:
                msg = log_q.get_nowait()
                log_lines.append(ansi_to_html(msg))
                table = self._to_rows(fw.memory)
                yield log_lines, logs_to_html(log_lines), table
            except queue.Empty:
                try:
                    final_table = done_q.get_nowait()
                    yield log_lines, logs_to_html(log_lines), final_table
                    return
                except queue.Empty:
                    table = self._to_rows(fw.memory)
                    yield log_lines, logs_to_html(log_lines), table
                    time.sleep(0.1)

    def on_select(self, evt: gr.SelectData):
        fw = self._framework()
        row = evt.index[0]

        if row < len(fw.memory):
            opp = fw.memory[row]
            assert fw.planner is not None
            fw.planner.notifier.notify(opp)
            return f"Alert sent for: {opp.deal.description[:60]}..."

        return "Invalid selection"

    def init_state(self):
        fw = self._framework()
        return [], "", self._to_rows(fw.memory)

    def launch(self):
        with gr.Blocks(title="The Price is Right", fill_width=True) as ui:
            state_logs = gr.State([])

            gr.Markdown(
                '<div style="text-align: center; font-size: 28px; font-weight: bold; margin: 20px 0;">The Price is Right - Demo</div>'
                '<div style="text-align: center; font-size: 16px; color: #666; margin-bottom: 20px;">Multi-Agent Deal Hunting System (Mock Version)</div>'
            )

            with gr.Row():
                deals_table = gr.Dataframe(
                    headers=["Product Description", "Price", "Estimate", "Discount", "URL"],
                    wrap=True,
                    column_widths=[6, 1, 1, 1, 3],
                    row_count=10,
                    col_count=5,
                    max_height=420,
                    interactive=False,
                )

            with gr.Row():
                with gr.Column(scale=1):
                    logs_html = gr.HTML(label="Agent Logs")
                with gr.Column(scale=1):
                    vec_plot = gr.Plot(value=build_plot(), show_label=False)

            ui.load(self.init_state, inputs=[], outputs=[state_logs, logs_html, deals_table])

            timer = gr.Timer(value=10, active=True)
            timer.tick(
                self.scan_stream,
                inputs=[state_logs],
                outputs=[state_logs, logs_html, deals_table],
            )

            selection_feedback = gr.Textbox(visible=False)
            deals_table.select(self.on_select, inputs=[], outputs=[selection_feedback])

        ui.launch(share=True, inbrowser=True)

In [ ]:
ui_app = DealHunterUI()
ui_app.launch()